# Event-level histograms and overweight protection

Plot the weighted and unweighted event-level $\hat{p}_{T}$ and vertex-z histograms produced by `processForestSimple.C`. The final sections inspect the gen/reco dijet $p_{T}^{ave}/\hat{p}_{T}$ maps, reproduce the discrete upper-tail overweight-protection threshold from `macro/plotMcClosures.C::plotOverweightProtection`, fit that threshold, and overlay the fits on the original maps.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.event_plots import (
    draw_event_histograms, draw_overweight_map, draw_overweight_protection,
    print_fit_summary, rejected_fraction_by_x,
)
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)

## Configuration

The defaults compare p-going, Pb-going, and the existing combined file for embedding. `FILE_STEM='jetId'` matches the current production outputs. Integral normalization makes direction-shape comparisons meaningful; use `'none'` to retain the stored weighted or unweighted yields. Weighted overlays intentionally disable grids, while `DRAW_GRID` controls both axes for the unweighted overlays and overweight-protection plots. `SAVE_PNG` optionally writes a PNG beside each PDF. `OVERWEIGHT_DIRECTION` independently selects the file used for the gen/reco diagnostic. Two-dimensional maps use the shared square-canvas, Bird-palette, logarithmic-z, and palette-geometry settings from `hist_analysis.python.root_style`.

In [ ]:
GENERATOR = 'embedding'  # embedding or pythia
DIRECTIONS = ('pgoing', 'Pbgoing', 'combined')
FILE_STEM = 'jetId'
NORMALIZATION = 'integral'  # none or integral
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'events'
DRAW_GRID = True
SAVE_PNG = False
CUT_FRACTION = 0.005  # upper 0.5% tail
OVERWEIGHT_DIRECTION = 'pgoing'  # pgoing or Pbgoing
DIRECTION_COMPARISON_TAG = 'directionComparison'

def mc_file(direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
    return resolve_direction_file(BASE_DIR, GENERATOR, direction, FILE_STEM)

files = {direction: mc_file(direction) for direction in DIRECTIONS}
missing = [str(path) for path in files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing configured ROOT files:\n' + '\n'.join(missing))
files

## Weighted event distributions

In [ ]:
weighted_canvases = {}
for key, x_title, log_y in (
    ('hPtHat', '#hat{p}_{T} (GeV)', True),
    ('hVz', 'v_{z} (cm)', False),
):
    histograms = {direction: load_histogram(str(path), key)
                  for direction, path in files.items()}
    canvas, plotted = draw_event_histograms(
        histograms, title=key, x_title=x_title,
        generator_label=GENERATOR,
        normalization=NORMALIZATION, log_y=log_y, grid=False,
        output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION_COMPARISON_TAG}_{key}.pdf',
        save_png=SAVE_PNG,
    )
    weighted_canvases[key] = (canvas, plotted)
    display(canvas)

## Unweighted event distributions

These use `hPtHatUnweighted` and `hVzUnweighted`, which contain event populations before cross-section/event weighting. The configured display normalization is still applied to the plotted clones.

In [ ]:
unweighted_canvases = {}
for key, x_title, log_y in (
    ('hPtHatUnweighted', '#hat{p}_{T} (GeV)', True),
    ('hVzUnweighted', 'v_{z} (cm)', False),
):
    histograms = {direction: load_histogram(str(path), key)
                  for direction, path in files.items()}
    canvas, plotted = draw_event_histograms(
        histograms, title=key, x_title=x_title,
        generator_label=GENERATOR,
        normalization=NORMALIZATION, log_y=log_y, grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{GENERATOR}_{DIRECTION_COMPARISON_TAG}_{key}.pdf',
        save_png=SAVE_PNG,
    )
    unweighted_canvases[key] = (canvas, plotted)
    display(canvas)

## Overweight-protection inputs

The two-dimensional distributions expose unusually large dijet momentum relative to the hard-scattering scale. The macro's threshold calculation excludes underflow/overflow and scans each Y projection from high to low until it accumulates `CUT_FRACTION` of the in-range weight.

In [ ]:
overweight_file = mc_file(OVERWEIGHT_DIRECTION)
overweight_histograms = {
    level: load_histogram(str(overweight_file), f'h{level}DijetPtAveOverPtHatVsPtHat')
    for level in ('Gen', 'Reco')
}
overweight_pass_histograms = {
    level: load_histogram(str(overweight_file), f'h{level}DijetPtAveOverPtHatVsPtHatPass')
    for level in ('Gen', 'Reco')
}

overweight_maps = {}
for level, histogram in overweight_histograms.items():
    canvas = draw_overweight_map(
        histogram, title=f'{level} dijets',
        generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
        grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_{level.lower()}_overweight_map.pdf',
        save_png=SAVE_PNG,
    )
    overweight_maps[level] = canvas
    display(canvas)

## Distributions passing overweight protection

These stored maps contain events that pass the production overweight selection. `processForestSimple.C` fills either pass map only when neither the Gen nor Reco dijet is classified as overweight. The production selection uses its compiled fit coefficients; changing `CUT_FRACTION` in this notebook does not change these stored maps.

In [ ]:
overweight_pass_maps = {}
for level, histogram in overweight_pass_histograms.items():
    canvas = draw_overweight_map(
        histogram,
        title=f'{level} dijets passing overweight selection',
        generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
        grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_{level.lower()}_overweight_map_pass.pdf',
        save_png=SAVE_PNG,
    )
    overweight_pass_maps[level] = canvas
    display(canvas)

## Fraction rejected by overweight protection

For each $\hat{p}_{T}$ bin, this plots $(I_{all}-I_{pass})/I_{all}$, where each integral covers the in-range $p_{T}^{ave}/\hat{p}_{T}$ bins. Because passing requires both Gen and Reco protection to pass, these are event-level selection losses conditional on the corresponding Gen or Reco dijet entering the all histogram; they are not independent Gen-only and Reco-only failure probabilities. The calculation uses stored event weights.

In [ ]:
rejected_fractions = {
    level: rejected_fraction_by_x(
        overweight_histograms[level], overweight_pass_histograms[level],
        name=f'h{level}OverweightRejectedFraction',
    )
    for level in ('Gen', 'Reco')
}
rejected_maximum = max(
    histogram.GetBinContent(bin_index) + histogram.GetBinError(bin_index)
    for histogram in rejected_fractions.values()
    for bin_index in range(1, histogram.GetNbinsX() + 1)
)
rejected_canvas, rejected_fraction_plots = draw_event_histograms(
    rejected_fractions,
    title='Overweight-selection losses',
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    x_title='#hat{p}_{T} (GeV)',
    y_title='Rejected fraction',
    y_range=(0.0, 1.15 * rejected_maximum if rejected_maximum > 0.0 else 1.0),
    normalization='none', grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_overweight_rejected_fraction.pdf',
    save_png=SAVE_PNG,
)
display(rejected_canvas)

## Upper-tail thresholds and fits

The points use the macro's discrete Y-bin-center threshold definition. Both curves use $p_{0}+p_{1}\exp(-p_{2}x)+p_{3}\exp(-p_{4}x)$ over 15–950 GeV. Empty $\hat{p}_{T}$ bins remain zero; ROOT excludes points outside the fit range and bins without usable uncertainties from the fit. The cell prints every fitted parameter and $\chi^{2}/\mathrm{NDF}$.

In [ ]:
threshold_canvas, thresholds, fits = draw_overweight_protection(
    overweight_histograms['Gen'], overweight_histograms['Reco'],
    cut_fraction=CUT_FRACTION, grid=DRAW_GRID,
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_overweight_threshold.pdf',
    save_png=SAVE_PNG,
)
print_fit_summary(fits)
display(threshold_canvas)

## Gen overweight map with fitted protection threshold

The fitted threshold is overlaid on the original two-dimensional gen distribution.

In [ ]:
gen_fit_overlay_canvas = draw_overweight_map(
    overweight_histograms['Gen'],
    title='Gen dijets with threshold fit',
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    fit=fits['Gen'], grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_gen_overweight_map_fit.pdf',
    save_png=SAVE_PNG,
)
display(gen_fit_overlay_canvas)

## Reco overweight map with fitted protection threshold

The fitted threshold is overlaid on the original two-dimensional reco distribution.

In [ ]:
reco_fit_overlay_canvas = draw_overweight_map(
    overweight_histograms['Reco'],
    title='Reco dijets with threshold fit',
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    fit=fits['Reco'], grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_reco_overweight_map_fit.pdf',
    save_png=SAVE_PNG,
)
display(reco_fit_overlay_canvas)

## Leading-jet overweight-protection inputs

These maps show leading-jet $p_T/\hat{p}_T$ versus $\hat{p}_T$. The same discrete `CUT_FRACTION` upper-tail definition and fit model used for the dijet distributions are applied independently to Gen and Reco leading jets.

In [ ]:
lead_jet_overweight_histograms = {
    level: load_histogram(str(overweight_file), f'h{level}LeadJetPtOverPtHatVsPtHat')
    for level in ('Gen', 'Reco')
}
lead_jet_overweight_maps = {}
for level, histogram in lead_jet_overweight_histograms.items():
    canvas = draw_overweight_map(
        histogram,
        title=f'{level} leading jets',
        generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
        y_title='p_{T}^{Lead}/#hat{p}_{T}',
        grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_{level.lower()}_lead_jet_overweight_map.pdf',
        save_png=SAVE_PNG,
    )
    lead_jet_overweight_maps[level] = canvas
    display(canvas)

## Leading-jet upper-tail thresholds and fits

The cell prints all fit parameters with their errors and $\chi^2/\mathrm{NDF}$.

In [ ]:
lead_jet_threshold_canvas, lead_jet_thresholds, lead_jet_fits = draw_overweight_protection(
    lead_jet_overweight_histograms['Gen'], lead_jet_overweight_histograms['Reco'],
    cut_fraction=CUT_FRACTION, grid=DRAW_GRID,
    observable_label='Leading-jet', name_prefix='LeadJet',
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_lead_jet_overweight_threshold.pdf',
    save_png=SAVE_PNG,
)
print_fit_summary(lead_jet_fits)
display(lead_jet_threshold_canvas)

## Gen leading-jet map with fitted threshold

In [ ]:
gen_lead_jet_fit_overlay_canvas = draw_overweight_map(
    lead_jet_overweight_histograms['Gen'],
    title='Gen leading jets with threshold fit',
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    y_title='p_{T}^{Lead}/#hat{p}_{T}',
    fit=lead_jet_fits['Gen'], grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_gen_lead_jet_overweight_map_fit.pdf',
    save_png=SAVE_PNG,
)
display(gen_lead_jet_fit_overlay_canvas)

## Reco leading-jet map with fitted threshold

In [ ]:
reco_lead_jet_fit_overlay_canvas = draw_overweight_map(
    lead_jet_overweight_histograms['Reco'],
    title='Reco leading jets with threshold fit',
    generator_label=GENERATOR, orientation_label=OVERWEIGHT_DIRECTION,
    y_title='p_{T}^{Lead}/#hat{p}_{T}',
    fit=lead_jet_fits['Reco'], grid=DRAW_GRID,
    output=OUTPUT_DIR / f'{GENERATOR}_{OVERWEIGHT_DIRECTION}_reco_lead_jet_overweight_map_fit.pdf',
    save_png=SAVE_PNG,
)
display(reco_lead_jet_fit_overlay_canvas)